# Enveda CASMI 2026 — MS/MS Mass Spectra Retrieval EDA, Offline RDKit & Dual-Band Starter

This public starter notebook provides an exploratory analysis for **Enveda CASMI 2026 (`enveda-CASMI26-molecule-id-mass-spectra`)**, combining:
- **Competition Input File Inspection** (`enveda-CASMI26-molecule-id-mass-spectra`)
- **Offline RDKit 2026.3.3 Installation Guide** (`thivnti/casmi2026-offline-rdkit`)
- **MS/MS Candidate Retrieval Evaluation Benchmark** (`thivnti/mass-spectrometry-retrieval-evaluation-benchmark`)
- **Dual-Band 4096-Bit Spectral Fingerprint Projector** (`thivnti/msms-spectral-fingerprint-projector`)

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def find_input_files(substring=None):
    matches = []
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        for fname in files:
            if substring is None or substring in fname:
                matches.append(Path(root) / fname)
    return sorted(matches)

all_files = find_input_files()
print(f"Total mounted files discovered: {len(all_files)}")
for p in all_files[:15]:
    print(f" - {p}")


In [ ]:
ms_csv = find_input_files('msms_retrieval_benchmark.csv')[0]
df_ms = pd.read_csv(ms_csv)
print(f'MS/MS Retrieval Benchmark Shape: {df_ms.shape}')
display(df_ms.head(10))

adduct_stats = df_ms.groupby(['ionization_mode', 'adduct_type']).agg(
    count=('spectrum_group_id', 'count'),
    mean_mrr=('reciprocal_rank', 'mean'),
    top1_rate=('reciprocal_rank', lambda s: np.mean(s >= 0.999)),
    mean_cosine=('cosine_similarity_top1', 'mean'),
    mean_neutral_loss=('neutral_loss_score', 'mean'),
    median_ppm_err=('precursor_ppm_error', 'median')
).reset_index().round(4)
display(adduct_stats)
adduct_stats.to_csv('casmi26_adduct_retrieval_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=130)
axes[0].bar(adduct_stats['adduct_type'], adduct_stats['mean_mrr'], color='#10b981')
axes[0].set_ylabel('Mean Reciprocal Rank (MRR@25)')
axes[0].set_title('MRR@25 by Precursor Adduct Species', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=25)

axes[1].scatter(df_ms['cosine_similarity_top1'], df_ms['neutral_loss_score'], c=df_ms['reciprocal_rank'], cmap='viridis', alpha=0.75)
axes[1].set_xlabel('Direct Cosine Similarity (Sqrt-L2)')
axes[1].set_ylabel('Neutral-Loss Cosine Score')
axes[1].set_title('Dual-Channel Spectral Similarity vs Reciprocal Rank', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('casmi26_msms_eda_dashboard.png', dpi=130)
plt.show()

In [ ]:
cfg_files = find_input_files('model_config.json')
if cfg_files:
    cfg = json.loads(cfg_files[0].read_text())
    print('Loaded Dual-Band 4096-Bit Projector Config:', json.dumps(cfg, indent=2))